# Explore one acquisition

HD-002 has about 20,000 gene-positive cells, small enough to load fully. This tutorial follows
the familiar single-cell workflow (QC → normalization → PCA → UMAP → clusters → markers)
and adds what spatial data offers: every cell has a position in microns.

> **Code-only page.** Outputs appear when you run the notebook on the downloaded sample.

> **Requirements.** `pip install scanpy anndata pyarrow pandas matplotlib igraph leidenalg` · HD-002 downloaded as in [Tutorial 1](01_download_and_read.ipynb).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt

SAMPLE_ROOT = Path("smcc_data/HD/samples/HD-002")
sample = pq.read_table(SAMPLE_ROOT / "metadata/samples.parquet").to_pylist()[0]

adata = ad.read_h5ad(SAMPLE_ROOT / sample["expression_file"])      # in memory, still sparse
genes = pq.read_table(SAMPLE_ROOT / sample["genes_file"]).to_pandas()
adata

## 1. Keep genes only

Attach the feature table to `adata.var`, then keep **Gene Expression** features. On Xenium this
step removes negative-control probes and codewords; skipping it inflates counts.

In [ ]:
genes.index = adata.var_names
adata.var = adata.var.join(genes, rsuffix="_feature")
symbol_col = next((c for c in genes.columns if "symbol" in c.lower()), "gene_id")
adata.var_names = adata.var[symbol_col].astype(str)
adata.var_names_make_unique()

adata = adata[:, adata.var["feature_type"] == "Gene Expression"].copy()
adata.layers["counts"] = adata.X.copy()        # keep raw integers
adata

## 2. Quality control

In [ ]:
sc.pp.calculate_qc_metrics(adata, inplace=True, percent_top=None, log1p=True)
sc.pl.violin(adata, ["total_counts", "n_genes_by_counts"], jitter=0.3, multi_panel=True)

Every cell in the release already has at least one positive gene count. Choose thresholds for
*your* question; the benchmark itself does not filter cells by depth.

In [ ]:
sc.pp.filter_cells(adata, min_counts=20)
sc.pp.filter_genes(adata, min_cells=10)
adata.n_obs, adata.n_vars

## 3. Cells in space

`obsm["spatial_um"]` gives each centroid in microns (`obsm["spatial"]` is in image pixels).

In [ ]:
xy = adata.obsm["spatial_um"]
fig, ax = plt.subplots(figsize=(7, 6))
sc_ = ax.scatter(xy[:, 0], xy[:, 1], c=adata.obs["log1p_total_counts"], s=1, cmap="magma", rasterized=True)
ax.invert_yaxis(); ax.set_aspect("equal"); ax.set_xlabel("x (µm)"); ax.set_ylabel("y (µm)")
fig.colorbar(sc_, label="log1p total counts"); ax.set_title(sample["sample_id"])
plt.show()

## 4. Normalize, reduce, cluster

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
sc.pp.pca(adata, n_comps=30, mask_var="highly_variable")
sc.pp.neighbors(adata, n_pcs=30)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.6, flavor="igraph", n_iterations=2)
sc.pl.umap(adata, color="leiden", legend_loc="on data", frameon=False)

The same clusters, placed back in the tissue:

In [ ]:
sc.pl.embedding(adata, basis="spatial_um", color="leiden", s=4, frameon=False)

## 5. Marker genes

In [ ]:
sc.tl.rank_genes_groups(adata, "leiden", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=3)

> **Cell-type labels.** HD cells also carry annotation *evidence* (marker support, prediction tiers, uncalibrated scores, or `Unresolved`). Treat it as evidence, not manually curated ground truth. See [Sample context](../dataset/context.md).

## Next

[Tutorial 3](03_histology_crops.ipynb) attaches each cell's own H&E crop.